### Bước 5: GỘP RASTER THEO MÙA VỤ 2020–2024 (Bắc – Trung – Nam)

In [1]:
import os
import rasterio
import numpy as np
from rasterio.mask import mask
from datetime import datetime
import geopandas as gpd
from tqdm import tqdm

In [ ]:


# ==========================================
# 1. INPUTS
# ==========================================
INPUT_DIR = r"E:\DownloadData\co2_ban_do\output_500m_align"
OUT_DIR = r"E:\DownloadData\co2_ban_do\output_seasonal_variable"
os.makedirs(OUT_DIR, exist_ok=True)

# shapefile vùng
RRD_SHAPE = r"E:\RanhGioi\DongBangSongHong\DongBangSongHong.shp"
MKD_SHAPE = r"E:\RanhGioi\DongBangSongCuuLong\DongBangSongCuuLong.shp"

regions = {
    "RRD": gpd.read_file(RRD_SHAPE).to_crs("EPSG:32648"),
    "MKD": gpd.read_file(MKD_SHAPE).to_crs("EPSG:32648"),
}

# ==========================================
# 2. SEASON DEFINITIONS
# ==========================================
SEASONS = {
    "RRD": {
        "ChiemXuan": ("12-01_prev", "05-15"),
        "Mua":  ("05-16", "11-15"),
    },
    "MKD": {
        "DongXuan": ("11-01_prev", "03-31"),
        "HeThu":    ("04-01", "08-15"),
        "ThuDong":  ("08-16", "10-31")
    }
}

# ==========================================
# 3. KHAI BÁO BAND CHO MỖI DATASET
# ==========================================
BANDS = {

    "ndvi_evi": {
        1: "NDVI",
        2: "EVI",
        3: "LSWI",
        4: "NDWI_McFeeters"
    },

    "era5": {
        1: "temperature_2m",
        2: "skin_temperature",
        3: "soil_temperature_L1",
        4: "soil_water_L1",
        5: "surface_solar_radiation",
        6: "total_precipitation",
        7: "lai_low_veg",
        8: "wind_u_10m",
        9: "wind_v_10m",
        10: "dewpoint_temp_2m",
        11: "surface_pressure"
    },

    "et_pet": {
        1: "ET",
        2: "LE",
        3: "PET",
        4: "PLE"
    },

    "fpar_lai": {
        1: "FPAR",
        2: "LAI"
    },

    "optical_depth": {
        1: "AOD_047um",
        2: "AOD_055um"
    },

    "par": {
        1: "GMT_0000_PAR",
        2: "GMT_0300_PAR",
        3: "GMT_0600_PAR",
        4: "GMT_0900_PAR"
    },

    "smap": {
        1: "sm_surface_daily"
    },

    "chirps": {
        1: "precipitation"
    }
}

# ==========================================
# 4. TÁCH NGÀY TỪ TÊN FILE
# ==========================================
def extract_date(name):
    parts = name.split("_")
    for p in parts:
        if len(p) == 10 and p[4] == "-" and p[7] == "-":
            try:
                return datetime.strptime(p, "%Y-%m-%d")
            except:
                pass
    return None

# ==========================================
# 5. GỘP BAND THEO MÙA
# ==========================================
def merge_band(files, band_idx, region_geom):
    stack = []
    meta = None

    for f in files:
        with rasterio.open(f) as src:

            if meta is None:
                meta = src.meta.copy()

            clipped, _ = mask(src, region_geom, crop=False, filled=False)
            arr = clipped[band_idx-1].filled(np.nan)

            stack.append(arr.astype("float32"))

    if len(stack) == 0:
        return None, None

    return np.nanmean(np.stack(stack), axis=0), meta

# ==========================================
# 6. MAIN LOOP
# ==========================================
for dataset_name, band_map in BANDS.items():

    dataset_dir = os.path.join(INPUT_DIR, dataset_name)
    if not os.path.isdir(dataset_dir):
        print("⛔ Missing dataset:", dataset_name)
        continue

    print("\n=========================")
    print("✔ Dataset:", dataset_name)
    print("=========================")

    for year_folder in os.listdir(dataset_dir):

        year_path = os.path.join(dataset_dir, year_folder)
        if not os.path.isdir(year_path):
            continue

        # TÁCH NĂM
        year = int("".join(filter(str.isdigit, year_folder))[:4])

        files = [f for f in os.listdir(year_path) if f.endswith(".tif")]
        if len(files) == 0:
            continue

        # DUYỆT REGION
        for region_name, gdf in regions.items():

            region_geom = [gdf.unary_union.__geo_interface__]

            # DUYỆT SEASON
            for season_name, (s0, s1) in SEASONS[region_name].items():

                if "prev" in s0:
                    start = datetime.strptime(f"{year-1}-{s0.replace('_prev','')}", "%Y-%m-%d")
                else:
                    start = datetime.strptime(f"{year}-{s0}", "%Y-%m-%d")

                if "prev" in s1:
                    end = datetime.strptime(f"{year-1}-{s1.replace('_prev','')}", "%Y-%m-%d")
                else:
                    end = datetime.strptime(f"{year}-{s1}", "%Y-%m-%d")

                # LỌC FILE THEO NGÀY
                season_files = []
                for fname in files:
                    d = extract_date(fname)
                    if d is not None and start <= d <= end:
                        season_files.append(os.path.join(year_path, fname))

                if len(season_files) == 0:
                    print(f"⚠ No data: {dataset_name} – {region_name} – {year} – {season_name}")
                    continue

                # GỘP TỪNG BAND
                for bidx, varname in band_map.items():

                    merged, meta = merge_band(season_files, bidx, region_geom)
                    if merged is None:
                        continue

                    out_folder = os.path.join(OUT_DIR, varname, region_name)
                    os.makedirs(out_folder, exist_ok=True)

                    out_path = os.path.join(out_folder, f"{varname}_{region_name}_{year}_{season_name}.tif")

                    meta.update({"count": 1, "dtype": "float32"})

                    with rasterio.open(out_path, "w", **meta) as dst:
                        dst.write(merged, 1)

                    print("✔ Saved:", out_path)



✔ Dataset: ndvi_evi


C:\Users\ASUS\AppData\Local\Temp/ipykernel_18116/3776205438.py:126: RuntimeWarning: Mean of empty slice
  return np.nanmean(np.stack(stack), axis=0), meta


✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_variable\NDVI\RRD\NDVI_RRD_2020_Xuan.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_variable\EVI\RRD\EVI_RRD_2020_Xuan.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_variable\LSWI\RRD\LSWI_RRD_2020_Xuan.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_variable\NDWI_McFeeters\RRD\NDWI_McFeeters_RRD_2020_Xuan.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_variable\NDVI\RRD\NDVI_RRD_2020_Mua.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_variable\EVI\RRD\EVI_RRD_2020_Mua.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_variable\LSWI\RRD\LSWI_RRD_2020_Mua.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_variable\NDWI_McFeeters\RRD\NDWI_McFeeters_RRD_2020_Mua.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_variable\NDVI\RRD\NDVI_RRD_2020_Dong.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_variable\EVI\RRD\EVI_RRD_2020_Dong.tif
✔ Saved: E:\DownloadData\co2_ban_do\outp

In [4]:
import os
import rasterio
from rasterio.mask import mask
import geopandas as gpd
import numpy as np

# ==========================================
# 1. INPUTS
# ==========================================
INPUT_DIR = r"E:\DownloadData\co2_ban_do\output_500m_align_soilgrids"
OUT_DIR = r"E:\DownloadData\co2_ban_do\output_seasonal_soilgrids"
os.makedirs(OUT_DIR, exist_ok=True)

# File soil cần xử lý
soil_files = {
    "soc_0_5":  "soilgrids-isric_myRegion_soc_0-5cm_mean_32648_500m_aligned.tif",
    "soc_5_15": "soilgrids-isric_myRegion_soc_5-15cm_mean_32648_500m_aligned.tif",
    "soc_15_30": "soilgrids-isric_myRegion_soc_15-30cm_mean_32648_500m_aligned.tif",
    "ph_0_5":   "soilgrids-isric_myRegion_phh2o_0-5cm_mean_32648_500m_aligned.tif",
    "ph_5_15":  "soilgrids-isric_myRegion_phh2o_5-15cm_mean_32648_500m_aligned.tif",
    "ph_15_30": "soilgrids-isric_myRegion_phh2o_15-30cm_mean_32648_500m_aligned.tif"
}

# Shapefile vùng
RRD_SHAPE = r"E:\RanhGioi\DongBangSongHong\DongBangSongHong.shp"
MKD_SHAPE = r"E:\RanhGioi\DongBangSongCuuLong\DongBangSongCuuLong.shp"

regions = {
    "RRD": gpd.read_file(RRD_SHAPE).to_crs("EPSG:32648"),
    "MKD": gpd.read_file(MKD_SHAPE).to_crs("EPSG:32648"),
}

# ==========================================
# 2. XỬ LÝ CLIP
# ==========================================
for soil_key, soil_filename in soil_files.items():

    soil_path = os.path.join(INPUT_DIR, soil_filename)

    if not os.path.exists(soil_path):
        print(f"⚠ FILE KHÔNG TỒN TẠI: {soil_path}")
        continue

    print(f"\n🔎 Đang xử lý: {soil_key}")

    with rasterio.open(soil_path) as src:
        soil_meta = src.meta.copy()

        for region_name, gdf in regions.items():

            geom = [gdf.unary_union.__geo_interface__]

            clipped, transform = mask(
                src,
                geom,
                crop=False,
                filled=False
            )

            # ⭐ SỬA LỖI DTYPE → CHUYỂN FLOAT32 TRƯỚC
            data = clipped[0].astype("float32")
            mask_arr = clipped[0].mask
            soil_arr = np.where(mask_arr, np.nan, data)

            out_meta = soil_meta.copy()
            out_meta.update({
                "count": 1,
                "dtype": "float32",
                "transform": transform
            })

            out_file = os.path.join(
                OUT_DIR, f"{soil_key}_{region_name}.tif"
            )

            with rasterio.open(out_file, "w", **out_meta) as dst:
                dst.write(soil_arr, 1)

            print(f"✔ Saved: {out_file}")

print("\n🎉 HOÀN TẤT TOÀN BỘ SOILGRIDS!")



🔎 Đang xử lý: soc_0_5
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_soilgrids\soc_0_5_RRD.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_soilgrids\soc_0_5_MKD.tif

🔎 Đang xử lý: soc_5_15
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_soilgrids\soc_5_15_RRD.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_soilgrids\soc_5_15_MKD.tif

🔎 Đang xử lý: soc_15_30
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_soilgrids\soc_15_30_RRD.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_soilgrids\soc_15_30_MKD.tif

🔎 Đang xử lý: ph_0_5
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_soilgrids\ph_0_5_RRD.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_soilgrids\ph_0_5_MKD.tif

🔎 Đang xử lý: ph_5_15
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_soilgrids\ph_5_15_RRD.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_soilgrids\ph_5_15_MKD.tif

🔎 Đang xử lý: ph_15_30
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_soilgrids\ph_15_30_RRD.

In [1]:
import os
import rasterio
import numpy as np
import geopandas as gpd
from rasterio.mask import mask
from tqdm import tqdm
from datetime import datetime

# ==========================================
# 1. INPUTS
# ==========================================
INPUT_DIR = r"E:\DownloadData\co2_ban_do\output_500m_align\era5"
OUT_DIR = r"E:\DownloadData\co2_ban_do\output_seasonal_variable"
os.makedirs(OUT_DIR, exist_ok=True)

# shapefile vùng
RRD_SHAPE = r"E:\RanhGioi\DongBangSongHong\DongBangSongHong.shp"
MKD_SHAPE = r"E:\RanhGioi\DongBangSongCuuLong\DongBangSongCuuLong.shp"

regions = {
    "RRD": gpd.read_file(RRD_SHAPE).to_crs("EPSG:32648"),
    "MKD": gpd.read_file(MKD_SHAPE).to_crs("EPSG:32648"),
}

# ==========================================
# 2. SEASONS DEFINED
# ==========================================
SEASONS = {
    "RRD": {
        "Xuan": ("01-01", "05-31"),
        "Mua":  ("06-01", "10-31"),
        "Dong": ("11-01", "12-31"),
    },
    "MKD": {
        "DongXuan": ("11-01_prev", "02-28"),
        "HeThu":    ("03-01", "06-30"),
        "ThuDong":  ("08-01", "10-31"),
    }
}

# ==========================================
# 3. BANDS ERA5 (theo đúng thứ tự trong raster)
# ==========================================
ERA5_BANDS = {
    1: "temperature_2m",
    2: "skin_temperature",
    3: "soil_temperature_L1",
    4: "soil_water_L1",
    5: "surface_solar_radiation",
    6: "total_precipitation",
    7: "lai_low_veg",
    8: "wind_u_10m",
    9: "wind_v_10m",
    10: "dewpoint_temp_2m",
    11: "surface_pressure",
}

# ==========================================
# 4. Extract date from filename ERA5
# ==========================================
def extract_date(filename):
    # tìm chuỗi YYYY-MM-DD
    name = os.path.basename(filename)
    parts = name.split("_")
    for p in parts:
        if len(p) == 10 and p[4] == "-" and p[7] == "-":
            try:
                return datetime.strptime(p, "%Y-%m-%d")
            except:
                pass
    return None

# ==========================================
# 5. Merge a band list
# ==========================================
def merge_band(files, bidx, region_geom):
    stack = []
    meta = None

    for f in files:
        with rasterio.open(f) as src:
            if meta is None:
                meta = src.meta.copy()

            clipped, _ = mask(src, region_geom, crop=False, filled=False)
            arr = clipped[bidx - 1].filled(np.nan)
            stack.append(arr.astype("float32"))

    if len(stack) == 0:
        return None, None

    return np.nanmean(np.stack(stack), axis=0), meta


# ==========================================
# 6. MAIN LOOP — PROCESS ERA5 BY FOLDER
# ==========================================
YEARS_TO_PROCESS = [2022, 2023, 2024]

for year_folder in os.listdir(INPUT_DIR):

    folder_path = os.path.join(INPUT_DIR, year_folder)
    if not os.path.isdir(folder_path):
        continue

    # lấy năm từ folder
    try:
        year = int(year_folder[-4:])
    except:
        print("❌ Không tách được năm từ folder:", year_folder)
        continue

    # 🚨 BỎ QUA NẾU NĂM KHÔNG NẰM TRONG LIST
    if year not in YEARS_TO_PROCESS:
        print(f"⏭ Bỏ qua {year}")
        continue

    print("\n==============================")
    print("✔ ERA5 YEAR:", year)
    print("==============================")


    # list file .tif
    files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith(".tif")]

    if len(files) == 0:
        print("⚠ No ERA5 files in:", folder_path)
        continue

    # ===============================
    # REGION LOOP
    # ===============================
    for region_name, gdf in regions.items():

        region_geom = [gdf.unary_union.__geo_interface__]

        # ===============================
        # SEASON LOOP
        # ===============================
        for season_name, (s0, s1) in SEASONS[region_name].items():

            # xử lý season bắt đầu từ năm trước
            if "prev" in s0:
                start = datetime.strptime(f"{year-1}-{s0.replace('_prev', '')}", "%Y-%m-%d")
            else:
                start = datetime.strptime(f"{year}-{s0}", "%Y-%m-%d")

            if "prev" in s1:
                end = datetime.strptime(f"{year-1}-{s1.replace('_prev', '')}", "%Y-%m-%d")
            else:
                end = datetime.strptime(f"{year}-{s1}", "%Y-%m-%d")

            # lọc file theo ngày
            season_files = []
            for fp in files:
                d = extract_date(fp)
                if d is not None and start <= d <= end:
                    season_files.append(fp)

            if len(season_files) == 0:
                print(f"⚠ No data: ERA5 – {region_name} – {year} – {season_name}")
                continue

            # ===============================
            # PROCESS EACH ERA5 BAND
            # ===============================
            for bidx, varname in ERA5_BANDS.items():

                merged, meta = merge_band(season_files, bidx, region_geom)
                if merged is None:
                    continue

                out_folder = os.path.join(OUT_DIR, varname, region_name)
                os.makedirs(out_folder, exist_ok=True)

                out_path = os.path.join(out_folder, f"{varname}_{region_name}_{year}_{season_name}.tif")

                meta.update({"count": 1, "dtype": "float32"})

                with rasterio.open(out_path, "w", **meta) as dst:
                    dst.write(merged, 1)

                print("✔ Saved:", out_path)

print("\n🎉 DONE — Tất cả ERA5 seasonal maps đã được tạo!")


⏭ Bỏ qua 2020
⏭ Bỏ qua 2021

✔ ERA5 YEAR: 2022


C:\Users\ASUS\AppData\Local\Temp/ipykernel_10856/3664408016.py:92: RuntimeWarning: Mean of empty slice
  return np.nanmean(np.stack(stack), axis=0), meta


✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_variable\temperature_2m\RRD\temperature_2m_RRD_2022_Xuan.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_variable\skin_temperature\RRD\skin_temperature_RRD_2022_Xuan.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_variable\soil_temperature_L1\RRD\soil_temperature_L1_RRD_2022_Xuan.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_variable\soil_water_L1\RRD\soil_water_L1_RRD_2022_Xuan.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_variable\surface_solar_radiation\RRD\surface_solar_radiation_RRD_2022_Xuan.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_variable\total_precipitation\RRD\total_precipitation_RRD_2022_Xuan.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_variable\lai_low_veg\RRD\lai_low_veg_RRD_2022_Xuan.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_variable\wind_u_10m\RRD\wind_u_10m_RRD_2022_Xuan.tif
✔ Saved: E:\DownloadData\co2_ban_do\output_seasonal_variable\wind_v_10